# Clase 1 · Gradient Descent desde cero con IA
### Curso 2: Machine Learning End-to-End con IA · Bloque 9
**PEDS – Promo 7** · Docente: **Josef Renato Rodriguez** · Miércoles 25/11/2026

---

En esta clase vamos a **abrir la caja negra** del entrenamiento de un modelo. Cuando llamas a `model.fit()`, por dentro casi siempre hay un algoritmo que *baja una montaña paso a paso* hasta encontrar los mejores parámetros. Ese algoritmo es el **descenso del gradiente (Gradient Descent)**, y hoy lo vamos a construir con nuestras propias manos… y con ayuda de cuatro asistentes de IA.

**Al terminar la clase podrás:**
1. Explicar qué es un modelo de regresión lineal y qué son sus parámetros $w$ y $b$.
2. Calcular la **función de costo** $J(w,b)$ y *verla* como una superficie en forma de tazón.
3. Entender el **gradiente** como la pendiente que nos dice hacia dónde bajar.
4. Implementar **Gradient Descent desde cero** y controlar su **learning rate** $\alpha$.
5. Extenderlo a **varias variables**, vectorizado y con **escalamiento de variables**.
6. Usar **Copilot, Claude, Codex y OpenCode** para generar código… y **auditarlo** antes de confiar en él.

> 💡 La secuencia de esta clase está inspirada en los labs del curso *Machine Learning Specialization* de Andrew Ng (DeepLearning.AI / Stanford), adaptada a un caso local y con visualizaciones nuevas e interactivas.

| Bloque | Tema | Tiempo |
|---|---|---|
| 0 | Setup y cómo trabajaremos con IA | 15' |
| 1–2 | El problema y el modelo $f_{w,b}(x)$ | 20' |
| 3 | Función de costo | 25' |
| 4 | Gradiente | 20' |
| 5 | Gradient Descent paso a paso | 25' |
| ☕ | Break | 10' |
| 6 | Learning rate | 15' |
| 7 | Regresión lineal múltiple vectorizada + escalamiento | 20' |
| 8 | Batch vs Mini-batch vs SGD | 10' |
| 9 | 🤖 Laboratorio con IA | 20' |

## 0. ¿Cómo vamos a usar la IA en este curso?

Usaremos cuatro herramientas. Cada una tiene un rol distinto:

| Herramienta | Dónde vive | Para qué la usaremos |
|---|---|---|
| **GitHub Copilot** | VS Code / Colab (autocompletado y chat) | Escribir código *mientras tecleas*: describes con un comentario y te sugiere la función. |
| **Claude** | claude.ai / Claude Code | Explicar conceptos, razonar sobre matemáticas, **revisar** código y encontrar bugs. |
| **Codex** | Agente de código (CLI / IDE) | Tareas más grandes en un repositorio: crear módulos, tests y ejecutarlos. |
| **OpenCode** | Agente de código open-source en la terminal | Lo mismo que un agente, pero eligiendo el modelo que quieras. Ideal para **comparar** respuestas. |

### 🧭 La regla de oro del curso

> **La IA escribe, tú verificas.** Ningún código generado por IA entra al notebook sin pasar una prueba.

Por eso, en cada sección vas a encontrar celdas **✅ Verificación** que comparan tu implementación (o la de la IA) con un resultado conocido.

⚠️ **Nunca** pegues datos reales de clientes, credenciales ni información confidencial de tu empresa en una herramienta de IA. En este curso usamos datos simulados.

### 0.1 Setup

Ejecuta esta celda. En **Google Colab** todo viene instalado; en tu máquina local instala con:

```bash
pip install numpy pandas matplotlib plotly ipywidgets scikit-learn
```

> Si usas **VS Code**, asegúrate de tener la extensión *Jupyter* y de que los widgets se muestren (los controles con sliders y botones).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from IPython.display import display, HTML, Markdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

%matplotlib inline
np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({
    "figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 12, "axes.titleweight": "bold",
})

# Paleta de la clase
AZUL, NARANJA, ROJO, VERDE, GRIS, MORADO = "#1f5fa8", "#f28e2b", "#d62728", "#2ca02c", "#7f7f7f", "#8e44ad"
print("✅ Librerías listas")

## 1. El problema: ¿cuánto cuesta un departamento?

Una inmobiliaria en Lima quiere estimar el **precio de venta** de un departamento a partir de su **área**. Tenemos 24 departamentos vendidos recientemente (datos simulados).

| Variable | Descripción |
|---|---|
| `area_m2` | Área del departamento en m² |
| `precio_miles_usd` | Precio de venta en miles de USD |

Para que los números sean cómodos, el modelo usará $x$ = **área en cientos de m²** (un depa de 100 m² → $x = 1.0$).

In [ ]:
rng = np.random.default_rng(7)
m = 24
area_m2 = np.round(rng.uniform(45, 160, m))
precio  = np.round(2.1 * area_m2 + 18 + rng.normal(0, 22, m), 1)

df = pd.DataFrame({"area_m2": area_m2, "precio_miles_usd": precio})

x_train = df["area_m2"].values / 100      # área en cientos de m²
y_train = df["precio_miles_usd"].values   # precio en miles de USD

print(f"m = {len(x_train)} ejemplos de entrenamiento")
df.head(8)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.scatter(x_train, y_train, c=AZUL, s=50, edgecolor="white", zorder=3)
ax.set_xlabel("x: área (cientos de m²)")
ax.set_ylabel("y: precio (miles USD)")
ax.set_title("Departamentos vendidos")
plt.show()

**Notación** que usaremos toda la clase (la misma de Andrew Ng):

| Símbolo | Significado | En Python |
|---|---|---|
| $x^{(i)}$ | variable de entrada del ejemplo $i$ | `x_train[i]` |
| $y^{(i)}$ | valor real (target) del ejemplo $i$ | `y_train[i]` |
| $m$ | número de ejemplos | `m` |
| $w$ | peso (pendiente) | `w` |
| $b$ | sesgo (intercepto) | `b` |
| $f_{w,b}(x^{(i)})$ | predicción del modelo para el ejemplo $i$ | `f_wb` |

## 2. El modelo: una recta

$$f_{w,b}(x) = w\,x + b$$

- $w$ controla la **inclinación**: cuántos miles de USD sube el precio por cada 100 m² adicionales.
- $b$ controla la **altura**: el precio "base".

Entrenar el modelo = **encontrar los valores de $w$ y $b$** que hacen que la recta pase lo más cerca posible de los puntos.

In [ ]:
def compute_model_output(x, w, b):
    '''Predicciones f_wb(x) = w*x + b para cada ejemplo.'''
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
    return f_wb

# versión vectorizada (la que usaremos en la práctica)
def predict(x, w, b):
    return w * x + b

print(compute_model_output(x_train[:3], 200, 10))
print(predict(x_train[:3], 200, 10))

### 🎨 Funciones de visualización

La siguiente celda define los gráficos interactivos de la clase. **No es necesario entenderla para seguir la clase**: ejecútala y sigue adelante (si te da curiosidad, pídele a Claude que te la explique 😉).

In [ ]:
# ============ Funciones de apoyo (referencia interna) ============
def _costo(x, y, w, b):
    return np.mean((w * x + b - y) ** 2) / 2

def _gradiente(x, y, w, b):
    err = w * x + b - y
    return np.mean(err * x), np.mean(err)

def _gd(x, y, w, b, alpha, iters):
    hist = [(w, b, _costo(x, y, w, b))]
    for _ in range(iters):
        dw, db = _gradiente(x, y, w, b)
        w, b = w - alpha * dw, b - alpha * db
        hist.append((w, b, _costo(x, y, w, b)))
    return np.array(hist)

# Solución exacta (mínimos cuadrados) para comparar
W_OPT, B_OPT = np.polyfit(x_train, y_train, 1)
J_OPT = _costo(x_train, y_train, W_OPT, B_OPT)

# Malla para superficies y contornos de J(w,b)
W_RANGO, B_RANGO = (-20, 420), (-220, 260)
_wg = np.linspace(*W_RANGO, 180)
_bg = np.linspace(*B_RANGO, 180)
WW, BB = np.meshgrid(_wg, _bg)
JJ = ((WW[..., None] * x_train + BB[..., None] - y_train) ** 2).mean(-1) / 2
NIVELES = np.logspace(np.log10(J_OPT * 1.02), np.log10(JJ.max()), 16)
XS = np.linspace(0.35, 1.7, 50)

def _recta(w, b, dec=0):
    return f"f(x) = {w:.{dec}f}·x {'+' if b >= 0 else '−'} {abs(b):.{dec}f}"

def _contorno(ax, titulo="Curvas de nivel de J(w,b)"):
    ax.contour(WW, BB, JJ, levels=NIVELES, cmap="viridis", linewidths=1)
    ax.plot(W_OPT, B_OPT, marker="*", ms=16, c=VERDE, mec="black", zorder=5)
    ax.set_xlim(*W_RANGO); ax.set_ylim(*B_RANGO)
    ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_title(titulo)

def _datos(ax, w, b, residuos=True):
    pred = w * x_train + b
    if residuos:
        ax.vlines(x_train, np.minimum(pred, y_train), np.maximum(pred, y_train),
                  color=ROJO, alpha=0.45, lw=1.5)
    ax.scatter(x_train, y_train, c=AZUL, s=40, edgecolor="white", zorder=3)
    ax.plot(XS, w * XS + b, c=NARANJA, lw=2.5)
    ax.set_xlim(0.35, 1.7); ax.set_ylim(-50, 450)
    ax.set_xlabel("x: área (cientos de m²)"); ax.set_ylabel("precio (miles USD)")

print(f"Solución exacta: w* = {W_OPT:.2f}, b* = {B_OPT:.2f}, J mínimo = {J_OPT:.2f}")

### 🎛️ Interactivo 1 · Ajusta la recta a mano

Mueve los sliders de $w$ y $b$. Las **líneas rojas** son los errores (residuos): la distancia entre el precio real y la predicción.

**Reto:** intenta llegar a un costo menor a 200 antes de presionar el botón 🎯.

In [ ]:
w_sl = widgets.FloatSlider(value=100, min=0, max=400, step=1, description="w", continuous_update=True, layout=Layout(width="420px"))
b_sl = widgets.FloatSlider(value=50, min=-150, max=250, step=1, description="b", continuous_update=True, layout=Layout(width="420px"))
btn_opt = widgets.Button(description="🎯 Mostrar la mejor recta", button_style="success", layout=Layout(width="220px"))
btn_rst = widgets.Button(description="⟲ Reiniciar", layout=Layout(width="120px"))

def _plot_modelo(w, b):
    J = _costo(x_train, y_train, w, b)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw={"width_ratios": [2.2, 1]})
    _datos(ax1, w, b)
    ax1.set_title(_recta(w, b))
    colores = [VERDE if J < 200 else (NARANJA if J < 1000 else ROJO), GRIS]
    ax2.bar(["Tu recta", "Mínimo\nposible"], [J, J_OPT], color=colores)
    ax2.set_yscale("log"); ax2.set_ylim(100, 1e5)
    ax2.set_title(f"Costo J = {J:,.1f}")
    for i, v in enumerate([J, J_OPT]):
        ax2.text(i, v * 1.15, f"{v:,.0f}", ha="center", fontweight="bold")
    plt.tight_layout(); plt.show()

btn_opt.on_click(lambda _: (setattr(w_sl, "value", round(W_OPT)), setattr(b_sl, "value", round(B_OPT))))
btn_rst.on_click(lambda _: (setattr(w_sl, "value", 100), setattr(b_sl, "value", 50)))
out1 = widgets.interactive_output(_plot_modelo, {"w": w_sl, "b": b_sl})
display(VBox([HBox([VBox([w_sl, b_sl]), VBox([btn_opt, btn_rst])]), out1]))

## 3. La función de costo $J(w,b)$

¿Cómo decide la computadora cuál recta es "mejor"? Necesita **un número** que mida qué tan mal lo está haciendo. Usamos el **error cuadrático medio** (dividido entre 2 por comodidad matemática):

$$J(w,b) = \frac{1}{2m} \sum_{i=0}^{m-1} \left(f_{w,b}(x^{(i)}) - y^{(i)}\right)^2$$

- Elevamos al cuadrado para que errores positivos y negativos no se cancelen y para castigar más los errores grandes.
- El $\frac{1}{2}$ hace que la derivada quede más limpia (el 2 del cuadrado se cancela).

**Objetivo del entrenamiento:** $\displaystyle \min_{w,b} J(w,b)$

In [ ]:
def compute_cost(x, y, w, b):
    '''
    Calcula el costo J(w,b) de la regresión lineal.
    x: (m,) entradas · y: (m,) targets · w, b: escalares
    '''
    m = x.shape[0]
    cost_sum = 0
    for i in range(m):
        f_wb = w * x[i] + b
        cost = (f_wb - y[i]) ** 2
        cost_sum = cost_sum + cost
    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost

def compute_cost_vec(x, y, w, b):
    '''Misma función, vectorizada con NumPy (sin bucles).'''
    return np.sum((w * x + b - y) ** 2) / (2 * x.shape[0])

print(f"J(100, 50)  = {compute_cost(x_train, y_train, 100, 50):,.2f}")
print(f"J(200, 10)  = {compute_cost(x_train, y_train, 200, 10):,.2f}")
print(f"Vectorizada = {compute_cost_vec(x_train, y_train, 200, 10):,.2f}")

In [ ]:
# ✅ Verificación: ¿bucle y vectorizado dan lo mismo? ¿cuál es más rápido?
import time
x_big = np.random.default_rng(0).uniform(0, 2, 200_000); y_big = 200 * x_big + 10
t0 = time.perf_counter(); c1 = compute_cost(x_big, y_big, 150, 0);     t1 = time.perf_counter()
c2 = compute_cost_vec(x_big, y_big, 150, 0);                          t2 = time.perf_counter()
print(f"Iguales: {np.isclose(c1, c2)} · bucle: {(t1-t0)*1000:.0f} ms · vectorizado: {(t2-t1)*1000:.1f} ms → {((t1-t0)/(t2-t1)):.0f}x más rápido")

### 🎛️ Interactivo 2 · El costo como función de $w$

Para entender la forma de $J$, primero **fijamos $b$** en su valor óptimo y movemos solo $w$. Observa cómo, a la derecha, el punto recorre una **parábola**: hay un único valor de $w$ que la hace mínima.

In [ ]:
w2_sl = widgets.FloatSlider(value=60, min=0, max=420, step=2, description="w", layout=Layout(width="520px"))
play2 = widgets.Play(value=0, min=0, max=420, step=6, interval=120, description="▶")
widgets.jslink((play2, "value"), (w2_sl, "value"))
_wline = np.linspace(0, 420, 200)
_Jline = np.array([_costo(x_train, y_train, w, B_OPT) for w in _wline])

def _plot_costo_w(w):
    J = _costo(x_train, y_train, w, B_OPT)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
    _datos(ax1, w, B_OPT); ax1.set_title(f"Modelo con w = {w:.0f}, b fijo = {B_OPT:.1f}")
    ax2.plot(_wline, _Jline, c=AZUL, lw=2)
    ax2.scatter([w], [J], s=120, c=NARANJA, edgecolor="black", zorder=5)
    ax2.axvline(W_OPT, ls="--", c=VERDE, alpha=0.7); ax2.text(W_OPT + 5, _Jline.max() * 0.9, "mínimo", color=VERDE)
    ax2.set_xlabel("w"); ax2.set_ylabel("J(w, b fijo)"); ax2.set_title(f"J = {J:,.1f}")
    plt.tight_layout(); plt.show()

out2 = widgets.interactive_output(_plot_costo_w, {"w": w2_sl})
display(VBox([HBox([play2, w2_sl]), out2]))

### 🌐 El tazón: $J(w,b)$ en 3D

Ahora dejamos que **ambos** parámetros varíen. La función de costo de la regresión lineal tiene forma de **tazón** (es *convexa*): un solo mínimo global, sin trampas.

👉 **Gira el gráfico con el mouse**, haz zoom y pasa el cursor para ver los valores. La ⭐ es el mínimo.

In [ ]:
fig3d = go.Figure()
fig3d.add_trace(go.Surface(x=_wg, y=_bg, z=JJ, colorscale="Viridis", opacity=0.92,
                           showscale=False, name="J(w,b)",
                           contours={"z": {"show": True, "usecolormap": True, "project": {"z": True}, "start": 0, "end": JJ.max(), "size": JJ.max() / 25}},
                           hovertemplate="w=%{x:.1f}<br>b=%{y:.1f}<br>J=%{z:,.0f}<extra></extra>"))
fig3d.add_trace(go.Scatter3d(x=[W_OPT], y=[B_OPT], z=[J_OPT], mode="markers",
                             marker=dict(size=7, color=VERDE, symbol="diamond"), name="mínimo"))
fig3d.update_layout(title="Función de costo J(w,b): el 'tazón'", height=560,
                    scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="J(w,b)",
                               camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))),
                    margin=dict(l=0, r=0, t=50, b=0))
fig3d.show()

### 🎛️ Interactivo 3 · Explora el mapa de curvas de nivel

Las **curvas de nivel** son el tazón visto desde arriba: cada anillo une puntos $(w,b)$ con el mismo costo (como un mapa topográfico). Mueve $w$ y $b$ y observa:
- Cerca de la ⭐ la recta se ajusta bien.
- Los anillos son **elipses alargadas y diagonales**: $w$ y $b$ están "acoplados" (si subes $w$, conviene bajar $b$). Volveremos a esto cuando hablemos de escalamiento.

In [ ]:
w3_sl = widgets.FloatSlider(value=320, min=W_RANGO[0], max=W_RANGO[1], step=2, description="w", layout=Layout(width="420px"))
b3_sl = widgets.FloatSlider(value=-150, min=B_RANGO[0], max=B_RANGO[1], step=2, description="b", layout=Layout(width="420px"))

def _plot_contorno_punto(w, b):
    J = _costo(x_train, y_train, w, b)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    _datos(ax1, w, b); ax1.set_title(f"{_recta(w, b)}   ·   J = {J:,.0f}")
    _contorno(ax2)
    ax2.scatter([w], [b], s=140, c=NARANJA, edgecolor="black", zorder=6)
    ax2.annotate(f"J = {J:,.0f}", (w, b), xytext=(10, 10), textcoords="offset points", fontweight="bold")
    plt.tight_layout(); plt.show()

out3 = widgets.interactive_output(_plot_contorno_punto, {"w": w3_sl, "b": b3_sl})
display(VBox([HBox([w3_sl, b3_sl]), out3]))

## 4. El gradiente: ¿hacia dónde bajar?

Imagina que estás en la montaña con neblina y solo puedes sentir la **inclinación del suelo bajo tus pies**. Para bajar, das un paso en la dirección donde el suelo desciende más. Esa inclinación es la **derivada** (con varias variables: el **gradiente**).

Para nuestro modelo, las derivadas parciales del costo son:

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)x^{(i)} \qquad\qquad \frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)$$

**Interpretación del signo:**
- Si $\frac{\partial J}{\partial w} > 0$ → el costo sube cuando $w$ sube → hay que **bajar** $w$.
- Si $\frac{\partial J}{\partial w} < 0$ → hay que **subir** $w$.

Por eso el paso siempre va en dirección **contraria** al gradiente: $w \leftarrow w - \alpha \frac{\partial J}{\partial w}$.

In [ ]:
def compute_gradient(x, y, w, b):
    '''
    Gradiente de J(w,b) para regresión lineal.
    Retorna: dj_dw, dj_db
    '''
    m = x.shape[0]
    dj_dw = 0
    dj_db = 0
    for i in range(m):
        f_wb = w * x[i] + b
        dj_dw_i = (f_wb - y[i]) * x[i]
        dj_db_i = f_wb - y[i]
        dj_db += dj_db_i
        dj_dw += dj_dw_i
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    return dj_dw, dj_db

print("En (w=100, b=50):", compute_gradient(x_train, y_train, 100, 50))
print("En el óptimo     :", compute_gradient(x_train, y_train, W_OPT, B_OPT), "← ¡casi cero!")

### ✅ Verificación: gradiente numérico

¿Cómo sabemos que la fórmula (o el código que nos dé una IA) está bien? Comparando con una **aproximación numérica** de la derivada:

$$\frac{\partial J}{\partial w} \approx \frac{J(w+\varepsilon, b) - J(w-\varepsilon, b)}{2\varepsilon}$$

Esta técnica (*gradient checking*) es la forma profesional de auditar cualquier implementación de gradientes.

In [ ]:
def gradiente_numerico(cost_fn, x, y, w, b, eps=1e-4):
    dw = (cost_fn(x, y, w + eps, b) - cost_fn(x, y, w - eps, b)) / (2 * eps)
    db = (cost_fn(x, y, w, b + eps) - cost_fn(x, y, w, b - eps)) / (2 * eps)
    return dw, db

for (w, b) in [(100, 50), (300, -100), (W_OPT, B_OPT)]:
    analitico = compute_gradient(x_train, y_train, w, b)
    numerico = gradiente_numerico(compute_cost, x_train, y_train, w, b)
    ok = np.allclose(analitico, numerico, atol=1e-4)
    print(f"(w={w:6.1f}, b={b:6.1f})  analítico={np.round(analitico, 4)}  numérico={np.round(numerico, 4)}  {'✅' if ok else '❌'}")

### 🎛️ Interactivo 4 · Descenso del gradiente en 1D (la pelotita que rueda)

Fijamos $b$ y aplicamos la regla $w \leftarrow w - \alpha\,\frac{\partial J}{\partial w}$ varias veces. La **línea punteada** es la tangente (la pendiente que "sentimos") y las **flechas** son los pasos.

Prueba:
1. $\alpha$ = 0.10 → pasos pequeñitos, llega lento.
2. $\alpha$ ≈ 0.80 → ¡llega casi en **un solo paso**!
3. $\alpha$ = 1.60 → rebota de lado a lado (zig-zag) pero converge.
4. $\alpha$ = 1.80 → 💥 **diverge**: cada rebote es más grande que el anterior.

Presiona ▶ para ver los pasos uno por uno.

In [ ]:
CURV_1D = np.mean(x_train ** 2)   # segunda derivada de J respecto de w
a4_sl = widgets.FloatSlider(value=0.10, min=0.05, max=2.0, step=0.05, description="α", readout_format=".2f", layout=Layout(width="360px"))
w04_sl = widgets.FloatSlider(value=40, min=0, max=420, step=10, description="w inicial", layout=Layout(width="360px"))
n4_sl = widgets.IntSlider(value=0, min=0, max=15, description="pasos", layout=Layout(width="360px"))
play4 = widgets.Play(value=0, min=0, max=15, step=1, interval=700)
widgets.jslink((play4, "value"), (n4_sl, "value"))
_w1d = np.linspace(W_OPT - 260, W_OPT + 260, 300)
_J1d = np.array([_costo(x_train, y_train, w, B_OPT) for w in _w1d])

def _plot_gd_1d(alpha, w0, n):
    ws = [w0]
    for _ in range(n):
        dw, _ = _gradiente(x_train, y_train, ws[-1], B_OPT)
        ws.append(ws[-1] - alpha * dw)
    ws = np.array(ws); Js = np.array([_costo(x_train, y_train, w, B_OPT) for w in ws])
    fig, ax = plt.subplots(figsize=(11, 4.6))
    ax.plot(_w1d, _J1d, c=AZUL, lw=2)
    visibles = np.abs(ws - W_OPT) < 260
    for i in range(len(ws) - 1):
        if visibles[i] and visibles[i + 1]:
            ax.annotate("", xy=(ws[i + 1], Js[i + 1]), xytext=(ws[i], Js[i]),
                        arrowprops=dict(arrowstyle="->", color=ROJO, lw=1.8))
    ax.scatter(ws[visibles], Js[visibles], c=NARANJA, s=60, edgecolor="black", zorder=5)
    wc = ws[-1]
    if abs(wc - W_OPT) < 260:
        pend, _ = _gradiente(x_train, y_train, wc, B_OPT); Jc = _costo(x_train, y_train, wc, B_OPT)
        tw = np.linspace(wc - 60, wc + 60, 10); ax.plot(tw, Jc + pend * (tw - wc), "--", c=MORADO, lw=1.5)
        txt = f"w = {wc:.1f} · pendiente dJ/dw = {pend:,.1f} · próximo paso = −α·pendiente = {-alpha*pend:,.1f}"
    else:
        txt = f"¡DIVERGE! w = {wc:,.0f} se salió del gráfico"
    ax.set_ylim(0, _J1d.max() * 1.05); ax.set_xlim(_w1d[0], _w1d[-1])
    ax.set_xlabel("w"); ax.set_ylabel("J(w, b fijo)")
    ax.set_title(f"α = {alpha:.2f}  ·  paso {n}  ·  {txt}", fontsize=10.5)
    plt.tight_layout(); plt.show()

out4 = widgets.interactive_output(_plot_gd_1d, {"alpha": a4_sl, "w0": w04_sl, "n": n4_sl})
display(VBox([HBox([a4_sl, w04_sl]), HBox([play4, n4_sl]), out4]))
print(f"Dato curioso: en este problema 1D, α = 1/mean(x²) = {1/CURV_1D:.2f} llega en 1 paso y α > 2/mean(x²) = {2/CURV_1D:.2f} diverge.")

### El campo de gradientes

En 2D, el gradiente $\left(\frac{\partial J}{\partial w}, \frac{\partial J}{\partial b}\right)$ es una flecha en cada punto del mapa. Graficamos **el negativo del gradiente** (la dirección en la que daremos el paso): todas las flechas "empujan" hacia el valle. Fíjate que no apuntan directo a la ⭐: primero caen al **valle diagonal** y luego avanzan por él. Eso explica el camino curvo que veremos en Gradient Descent.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
_contorno(ax, "Dirección de descenso: −∇J(w,b)")
wq, bq = np.meshgrid(np.linspace(0, 400, 14), np.linspace(-200, 240, 14))
U, V = np.zeros_like(wq), np.zeros_like(bq)
for i in range(wq.shape[0]):
    for j in range(wq.shape[1]):
        dw, db = compute_gradient(x_train, y_train, wq[i, j], bq[i, j])
        U[i, j], V[i, j] = -dw, -db
norma = np.sqrt(U ** 2 + V ** 2) + 1e-9
ax.quiver(wq, bq, U / norma, V / norma, norma, cmap="autumn_r", scale=28, width=0.004)
plt.show()

## 5. Gradient Descent: el algoritmo completo

$$
\begin{aligned}
&\text{repetir hasta converger: } \{ \\
&\qquad w \leftarrow w - \alpha \, \frac{\partial J(w,b)}{\partial w} \\
&\qquad b \leftarrow b - \alpha \, \frac{\partial J(w,b)}{\partial b} \\
&\}
\end{aligned}
$$

⚠️ **Actualización simultánea:** primero se calculan **ambas** derivadas con los valores actuales de $w$ y $b$, y *recién después* se actualizan los dos. Es un error clásico (¡y la IA también lo comete!) actualizar $w$ y usar el $w$ nuevo para calcular la derivada de $b$.

In [ ]:
import math

def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function, verbose=True):
    '''
    Ejecuta gradient descent para ajustar w y b.
    Retorna: w, b, J_history (costos), p_history ([w, b] por iteración)
    '''
    w, b = w_in, b_in
    J_history, p_history = [cost_function(x, y, w, b)], [[w, b]]
    for i in range(num_iters):
        # 1) calcular el gradiente con los valores ACTUALES
        dj_dw, dj_db = gradient_function(x, y, w, b)
        # 2) actualizar simultáneamente
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        # 3) guardar historia
        J_history.append(cost_function(x, y, w, b))
        p_history.append([w, b])
        if verbose and i % math.ceil(num_iters / 10) == 0:
            print(f"Iter {i:5d}: J = {J_history[-1]:10.2f} | dj_dw = {dj_dw:9.3f}, dj_db = {dj_db:9.3f} | w = {w:8.3f}, b = {b:8.3f}")
    return w, b, J_history, p_history

w_final, b_final, J_hist, p_hist = gradient_descent(
    x_train, y_train, w_in=0, b_in=0, alpha=0.5, num_iters=1000,
    cost_function=compute_cost, gradient_function=compute_gradient)

print(f"\n(w, b) encontrados por GD : ({w_final:.4f}, {b_final:.4f})")
print(f"(w, b) solución exacta    : ({W_OPT:.4f}, {B_OPT:.4f})")

In [ ]:
# Usemos el modelo entrenado
for area in [60, 100, 150]:
    print(f"Depa de {area} m² → precio estimado: {w_final * area/100 + b_final:,.1f} mil USD")

### 🎬 Animación · Mira al algoritmo aprender

Presiona **▶ Play** (o arrastra el slider de abajo). Los tres paneles se mueven juntos:
1. **El camino** de $(w,b)$ sobre el mapa de costo.
2. **La recta** acomodándose a los datos.
3. **La curva de aprendizaje**: el costo bajando iteración a iteración.

In [ ]:
def animar_gd(p_history, J_history, titulo="Gradient Descent en acción", n_frames=60):
    P = np.array(p_history); Jh = np.array(J_history); T = len(P) - 1
    idx = np.unique(np.round(np.geomspace(1, T + 1, n_frames)).astype(int) - 1)  # más frames al inicio
    fig = make_subplots(rows=1, cols=3, column_widths=[0.38, 0.34, 0.28],
                        subplot_titles=("Camino sobre J(w,b)", "Ajuste del modelo", "Curva de aprendizaje"))
    fig.add_trace(go.Contour(x=_wg, y=_bg, z=np.log10(JJ), colorscale="Viridis", showscale=False,
                             contours=dict(coloring="lines"), line_width=1.2, hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=[W_OPT], y=[B_OPT], mode="markers", marker=dict(symbol="star", size=16, color=VERDE, line=dict(width=1)), name="óptimo"), 1, 1)
    fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", marker=dict(color=AZUL, size=8), name="datos"), 1, 2)
    # trazas dinámicas (índices 3..6)
    fig.add_trace(go.Scatter(x=P[:1, 0], y=P[:1, 1], mode="lines+markers", line=dict(color=ROJO), marker=dict(size=4), name="camino"), 1, 1)
    fig.add_trace(go.Scatter(x=XS, y=P[0, 0] * XS + P[0, 1], mode="lines", line=dict(color=NARANJA, width=3), name="modelo"), 1, 2)
    fig.add_trace(go.Scatter(x=[0], y=Jh[:1], mode="lines", line=dict(color=MORADO), name="J"), 1, 3)
    fig.add_trace(go.Scatter(x=[0], y=Jh[:1], mode="markers", marker=dict(color=MORADO, size=10), name="J actual"), 1, 3)
    frames = []
    for k in idx:
        frames.append(go.Frame(name=str(k), traces=[3, 4, 5, 6], data=[
            go.Scatter(x=P[:k + 1, 0], y=P[:k + 1, 1]),
            go.Scatter(x=XS, y=P[k, 0] * XS + P[k, 1]),
            go.Scatter(x=np.arange(k + 1), y=Jh[:k + 1]),
            go.Scatter(x=[k], y=[Jh[k]])],
            layout=go.Layout(title_text=f"{titulo} · iteración {k} · w = {P[k,0]:.1f}, b = {P[k,1]:.1f}, J = {Jh[k]:,.1f}")))
    fig.frames = frames
    fig.update_layout(
        title=f"{titulo} · iteración 0", height=540, showlegend=False, margin=dict(t=90, b=140),
        updatemenus=[dict(type="buttons", direction="left", x=0.0, y=-0.3, xanchor="left", buttons=[
            dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=120, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
            dict(label="⏸ Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(x=0.18, y=-0.24, len=0.82, currentvalue=dict(prefix="iteración: "), steps=[
            dict(label=str(k), method="animate", args=[[str(k)], dict(frame=dict(duration=0, redraw=True), mode="immediate")]) for k in idx])])
    fig.update_xaxes(range=W_RANGO, title_text="w", row=1, col=1); fig.update_yaxes(range=B_RANGO, title_text="b", row=1, col=1)
    fig.update_xaxes(range=[0.35, 1.7], title_text="x (cientos de m²)", row=1, col=2); fig.update_yaxes(range=[-50, 450], row=1, col=2)
    fig.update_xaxes(range=[0, T], title_text="iteración", row=1, col=3)
    fig.update_yaxes(type="log", range=[np.log10(max(Jh.min(), 1)) - 0.1, np.log10(Jh.max()) + 0.1], title_text="J", row=1, col=3)
    return fig

animar_gd(p_hist[:301], J_hist[:301], titulo="GD con α = 0.5").show()

### 🎛️ Interactivo 5 · Tú eres el algoritmo: paso a paso

Ahora controlas cada iteración con los botones. Debajo del gráfico verás **la aritmética exacta** de cada paso: el gradiente, el tamaño del paso y los nuevos valores. Ideal para entender qué pasa "por dentro".

- **▶ 1 paso / ⏩ 10 pasos / ⏭ 100 pasos** avanzan el algoritmo.
- Cambia el **punto de partida** o el **α** y presiona **⟲ Reiniciar**.

In [ ]:
estado = {"hist": [], "ultimo": None}
a5_dd = widgets.Dropdown(options=[0.05, 0.1, 0.3, 0.5, 0.9], value=0.5, description="α", layout=Layout(width="160px"))
inicio_dd = widgets.Dropdown(options={"(0, 0)": (0., 0.), "(400, -200)": (400., -200.), "(50, 240)": (50., 240.), "(380, 200)": (380., 200.)},
                             value=(0., 0.), description="inicio", layout=Layout(width="200px"))
b1, b10, b100 = [widgets.Button(description=d, button_style="primary", layout=Layout(width="110px")) for d in ["▶ 1 paso", "⏩ 10 pasos", "⏭ 100 pasos"]]
brst = widgets.Button(description="⟲ Reiniciar", button_style="warning", layout=Layout(width="110px"))
out5 = widgets.Output(); info5 = widgets.HTML()

def _reiniciar(_=None):
    w0, b0 = inicio_dd.value
    estado["hist"] = [(w0, b0, _costo(x_train, y_train, w0, b0))]; estado["ultimo"] = None
    _dibujar5()

def _avanzar(n):
    alpha = a5_dd.value
    for _ in range(n):
        w, b, _J = estado["hist"][-1]
        dw, db = _gradiente(x_train, y_train, w, b)
        wn, bn = w - alpha * dw, b - alpha * db
        estado["hist"].append((wn, bn, _costo(x_train, y_train, wn, bn)))
        estado["ultimo"] = dict(w=w, b=b, dw=dw, db=db, wn=wn, bn=bn, alpha=alpha)
    _dibujar5()

def _dibujar5():
    H = np.array(estado["hist"]); k = len(H) - 1; w, b, J = H[-1]
    with out5:
        out5.clear_output(wait=True)
        fig, axs = plt.subplots(1, 3, figsize=(15, 4.3), gridspec_kw={"width_ratios": [1.15, 1.15, 0.9]})
        _contorno(axs[0], f"Camino · iteración {k}")
        axs[0].plot(H[:, 0], H[:, 1], "-o", c=ROJO, ms=3, lw=1.3)
        axs[0].scatter([w], [b], s=130, c=NARANJA, edgecolor="black", zorder=6)
        _datos(axs[1], w, b); axs[1].set_title(_recta(w, b, 1))
        axs[2].plot(H[:, 2], c=MORADO, lw=2); axs[2].scatter([k], [J], c=MORADO, s=50, zorder=5)
        axs[2].set_yscale("log"); axs[2].set_xlim(0, max(k, 10)); axs[2].set_xlabel("iteración"); axs[2].set_title(f"J = {J:,.2f}")
        axs[2].axhline(J_OPT, ls="--", c=VERDE, alpha=0.7)
        plt.tight_layout(); plt.show()
    u = estado["ultimo"]
    if u is None:
        info5.value = "<b>Presiona ▶ para dar el primer paso.</b>"
    else:
        info5.value = (
            f"<div style='font-family:monospace;font-size:13px;line-height:1.6;background:#f4f6fb;padding:8px 12px;border-radius:6px'>"
            f"<b>Último paso (iteración {k}) con α = {u['alpha']}</b><br>"
            f"∂J/∂w = {u['dw']:,.3f} &nbsp;&nbsp;&nbsp; ∂J/∂b = {u['db']:,.3f}<br>"
            f"w ← {u['w']:,.3f} − {u['alpha']} × ({u['dw']:,.3f}) = <b>{u['wn']:,.3f}</b><br>"
            f"b ← {u['b']:,.3f} − {u['alpha']} × ({u['db']:,.3f}) = <b>{u['bn']:,.3f}</b><br>"
            f"Distancia al óptimo: Δw = {u['wn']-W_OPT:,.3f}, Δb = {u['bn']-B_OPT:,.3f}</div>")

b1.on_click(lambda _: _avanzar(1)); b10.on_click(lambda _: _avanzar(10)); b100.on_click(lambda _: _avanzar(100))
brst.on_click(_reiniciar)
display(VBox([HBox([a5_dd, inicio_dd, b1, b10, b100, brst]), out5, info5]))
_reiniciar()

## 6. El learning rate $\alpha$: el hiperparámetro más importante

$\alpha$ define **el tamaño del paso**:

| α | Qué pasa |
|---|---|
| Muy pequeño | Converge, pero **muy lento** (desperdicias cómputo). |
| Adecuado | Converge rápido y de forma estable. |
| Grande | Converge con **zig-zag** (rebota entre las paredes del valle). |
| Demasiado grande | **Diverge**: el costo explota hacia infinito. |

### 🎛️ Interactivo 6 · Compara learning rates

Elige un α con los botones y presiona ▶. A la derecha, la curva de aprendizaje está en **escala logarítmica**: si sube en lugar de bajar, algo anda mal.

In [ ]:
ALFAS = [0.01, 0.1, 0.5, 0.9, 0.95]
N6 = 150
HIST6 = {a: _gd(x_train, y_train, 0., 0., a, N6) for a in ALFAS}
a6_tb = widgets.ToggleButtons(options=[(f"α = {a}", a) for a in ALFAS], value=0.5, description="")
it6_sl = widgets.IntSlider(value=N6, min=0, max=N6, description="iteración", layout=Layout(width="460px"))
play6 = widgets.Play(value=0, min=0, max=N6, step=2, interval=80)
widgets.jslink((play6, "value"), (it6_sl, "value"))

def _plot_lr(alpha, it):
    H = HIST6[alpha][: it + 1]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
    _contorno(ax1, f"Camino con α = {alpha}")
    dentro = (H[:, 0] > W_RANGO[0]) & (H[:, 0] < W_RANGO[1]) & (H[:, 1] > B_RANGO[0]) & (H[:, 1] < B_RANGO[1])
    ultimo_dentro = np.argmin(dentro) if not dentro.all() else len(H)
    ax1.plot(H[:ultimo_dentro, 0], H[:ultimo_dentro, 1], "-o", c=ROJO, ms=3, lw=1.2)
    if ultimo_dentro < len(H):
        ax1.text(0.5, 0.5, "¡DIVERGE!", transform=ax1.transAxes, ha="center", fontsize=26, color=ROJO, fontweight="bold", alpha=0.8)
    for a in ALFAS:
        Ha = HIST6[a][: it + 1, 2]
        ax2.plot(np.clip(Ha, 1, 1e8), lw=3 if a == alpha else 1.2, alpha=1 if a == alpha else 0.35, label=f"α = {a}")
    ax2.axhline(J_OPT, ls="--", c=VERDE); ax2.set_yscale("log"); ax2.set_ylim(100, 1e8); ax2.set_xlim(0, N6)
    ax2.set_xlabel("iteración"); ax2.set_ylabel("J (escala log)")
    ax2.set_title(f"Curva de aprendizaje · J actual = {H[-1, 2]:,.1f}" if H[-1, 2] < 1e8 else "Curva de aprendizaje · J → ∞")
    ax2.legend(loc="upper right", fontsize=9)
    plt.tight_layout(); plt.show()

out6 = widgets.interactive_output(_plot_lr, {"alpha": a6_tb, "it": it6_sl})
display(VBox([a6_tb, HBox([play6, it6_sl]), out6]))

**💬 Pregunta para la clase:** ¿por qué α = 0.9 converge y α = 0.95 diverge, si son tan parecidos?

La respuesta está en la **curvatura** del tazón. Para la regresión lineal, la matriz de segundas derivadas (Hessiana) es $H = \frac{1}{m}X^\top X$ (con una columna de unos para $b$). Gradient descent converge solo si:

$$\alpha < \frac{2}{\lambda_{\max}(H)}$$

donde $\lambda_{\max}$ es el autovalor más grande. Comprobémoslo:

In [ ]:
X_aug = np.c_[x_train, np.ones(m)]
H = X_aug.T @ X_aug / m
autovalores = np.linalg.eigvalsh(H)
print(f"Autovalores de H: {autovalores}")
print(f"α máximo para converger = 2 / λmax = {2 / autovalores.max():.4f}")
print(f"Número de condición λmax/λmin = {autovalores.max()/autovalores.min():.1f}  ← qué tan 'alargado' es el tazón")

> 🔑 **Idea clave:** el número de condición mide qué tan **alargadas** son las elipses. Cuanto más alargadas, más lento converge GD (el α queda limitado por la dirección empinada, pero hay que recorrer la dirección plana). La solución práctica es **escalar las variables**, que es justo lo que veremos ahora.

## 7. Regresión lineal múltiple: vectorizada y con escalamiento

En la vida real el precio depende de varias variables. Ahora tenemos 200 departamentos con:

| Variable | Descripción | Rango |
|---|---|---|
| `area_m2` | Área en m² | 40 – 180 |
| `dormitorios` | Número de dormitorios | 1 – 5 |
| `antiguedad` | Años del edificio | 0 – 40 |

El modelo pasa a ser:

$$f_{\mathbf{w},b}(\mathbf{x}) = w_1x_1 + w_2x_2 + w_3x_3 + b = \mathbf{w}\cdot\mathbf{x} + b$$

Con NumPy lo escribimos de forma **vectorizada** para todos los ejemplos a la vez: $\hat{\mathbf{y}} = X\mathbf{w} + b$.

In [ ]:
rng2 = np.random.default_rng(11)
n = 200
area = rng2.uniform(40, 180, n).round()
dormitorios = np.clip(np.round(area / 45 + rng2.normal(0, 0.6, n)), 1, 5)
antiguedad = rng2.uniform(0, 40, n).round()
precio2 = 1.9 * area + 12 * dormitorios - 1.6 * antiguedad + 40 + rng2.normal(0, 20, n)

df2 = pd.DataFrame({"area_m2": area, "dormitorios": dormitorios, "antiguedad": antiguedad, "precio_miles_usd": precio2.round(1)})
X_train = df2[["area_m2", "dormitorios", "antiguedad"]].values
y2_train = df2["precio_miles_usd"].values
FEATURES = ["area_m2", "dormitorios", "antiguedad"]
print("X_train:", X_train.shape, "· y:", y2_train.shape)
df2.describe().round(1)

In [ ]:
def compute_cost_multi(X, y, w, b):
    err = X @ w + b - y
    return (err @ err) / (2 * X.shape[0])

def compute_gradient_multi(X, y, w, b):
    m = X.shape[0]
    err = X @ w + b - y           # (m,)
    dj_dw = X.T @ err / m         # (n,)
    dj_db = err.mean()            # escalar
    return dj_dw, dj_db

def gradient_descent_multi(X, y, w_in, b_in, alpha, num_iters):
    w, b = w_in.copy(), b_in
    J_history = [compute_cost_multi(X, y, w, b)]
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradient_multi(X, y, w, b)
        w, b = w - alpha * dj_dw, b - alpha * dj_db
        J_history.append(compute_cost_multi(X, y, w, b))
    return w, b, np.array(J_history)

# Referencia: solución de scikit-learn
lr = LinearRegression().fit(X_train, y2_train)
W2_OPT, B2_OPT = lr.coef_, lr.intercept_
J2_OPT = compute_cost_multi(X_train, y2_train, W2_OPT, B2_OPT)
print("sklearn  → w =", W2_OPT.round(3), " b =", round(B2_OPT, 3), " J =", round(J2_OPT, 2))

### Intento 1: GD con las variables tal cual

`area_m2` vale ~100 y `dormitorios` ~2: la escala es 50 veces distinta. Mira lo que pasa:

In [ ]:
w_sin, b_sin, J_sin = gradient_descent_multi(X_train, y2_train, np.zeros(3), 0., alpha=1e-4, num_iters=1000)
print("GD sin escalar (α=1e-4, 1000 iter) → w =", w_sin.round(3), " b =", round(b_sin, 3), " J =", round(J_sin[-1], 2))
print("Óptimo (sklearn)                    → w =", W2_OPT.round(3), " b =", round(B2_OPT, 3), " J =", round(J2_OPT, 2))
w_x, b_x, J_x = gradient_descent_multi(X_train, y2_train, np.zeros(3), 0., alpha=2e-4, num_iters=50)
print("Y si subimos α a 2e-4 → J después de 50 iter =", f"{J_x[-1]:.2e}", "💥")

Con α = 1e-4 **no llega**, y con α = 2e-4 **explota**. El problema es la escala: el tazón es tan alargado que no hay un α que funcione bien en todas las direcciones.

### La solución: escalamiento z-score

$$x_j^{(i)} \leftarrow \frac{x_j^{(i)} - \mu_j}{\sigma_j}$$

Cada variable queda con media 0 y desviación 1, y el tazón se vuelve casi **redondo**.

⚠️ En un proyecto real, $\mu$ y $\sigma$ se calculan **solo con el set de entrenamiento** y se reutilizan para test y producción (si no, hay *data leakage*).

In [ ]:
def zscore_normalize(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    return (X - mu) / sigma, mu, sigma

X_norm, mu, sigma = zscore_normalize(X_train)
print("μ =", mu.round(2), "\nσ =", sigma.round(2))

def a_unidades_originales(w_norm, b_norm, mu, sigma):
    '''Convierte los parámetros entrenados con datos escalados a las unidades originales.'''
    w = w_norm / sigma
    b = b_norm - np.sum(w_norm * mu / sigma)
    return w, b

w_n, b_n, J_n = gradient_descent_multi(X_norm, y2_train, np.zeros(3), 0., alpha=0.5, num_iters=100)
w_orig, b_orig = a_unidades_originales(w_n, b_n, mu, sigma)
print("\nGD escalado (α=0.5, 100 iter) → w =", w_orig.round(3), " b =", round(b_orig, 3), " J =", round(J_n[-1], 2))
print("Óptimo (sklearn)               → w =", W2_OPT.round(3), " b =", round(B2_OPT, 3), " J =", round(J2_OPT, 2))

In [ ]:
# ¿Por qué funciona? Comparemos las curvas de nivel de J en el plano (w_area, w_antiguedad)
def _malla_multi(X, w_centro, b_centro, i, j, ri, rj, k=120):
    gi, gj = np.linspace(w_centro[i] - ri, w_centro[i] + ri, k), np.linspace(w_centro[j] - rj, w_centro[j] + rj, k)
    Z = np.zeros((k, k))
    for a, wj in enumerate(gj):
        for c, wi in enumerate(gi):
            w = w_centro.copy(); w[i], w[j] = wi, wj
            Z[a, c] = compute_cost_multi(X, y2_train, w, b_centro)
    return gi, gj, Z

w_norm_opt = W2_OPT * sigma; b_norm_opt = B2_OPT + np.sum(W2_OPT * mu)
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, (X, wc, bc, ri, rj, tit) in zip(axs, [
        (X_train, W2_OPT, B2_OPT, 1.0, 1.0, "SIN escalar: elipses muy alargadas"),
        (X_norm, w_norm_opt, b_norm_opt, 60, 60, "CON z-score: casi círculos")]):
    gi, gj, Z = _malla_multi(X, wc.copy(), bc, 0, 2, ri, rj)
    ax.contour(gi, gj, Z, levels=np.geomspace(Z.min() * 1.01, Z.max(), 15), cmap="viridis")
    ax.plot(wc[0], wc[2], "*", ms=15, c=VERDE, mec="black")
    ax.set_xlabel("w_area"); ax.set_ylabel("w_antiguedad"); ax.set_title(tit); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### 🎛️ Interactivo 7 · Laboratorio de escalamiento

Activa o desactiva el escalamiento, mueve α (escala logarítmica) y el número de iteraciones. A la derecha ves cada parámetro de GD (en unidades originales) como **% del valor exacto**: 🟩 llegó, 🟧 cerca, 🟥 lejos.

**Reto:** sin escalar, encuentra el mejor α posible. ¿Llegas al óptimo en 2 000 iteraciones?

In [ ]:
esc_tb = widgets.ToggleButtons(options=["Sin escalar", "Z-score"], value="Sin escalar")
a7_sl = widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=0, step=0.1, description="α", readout_format=".1e", layout=Layout(width="420px"))
it7_sl = widgets.IntSlider(value=300, min=10, max=2000, step=10, description="iteraciones", layout=Layout(width="420px"))
esc_tb.observe(lambda ch: setattr(a7_sl, "value", 1e-4 if ch["new"] == "Sin escalar" else 0.1), names="value")

def _plot_escalado(escala, alpha, iters):
    X = X_train if escala == "Sin escalar" else X_norm
    with np.errstate(all="ignore"):
        w, b, Jh = gradient_descent_multi(X, y2_train, np.zeros(3), 0., alpha, iters)
    if escala == "Z-score":
        w, b = a_unidades_originales(w, b, mu, sigma)
    diverge = not np.isfinite(Jh[-1]) or Jh[-1] > Jh[0]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.3))
    ax1.plot(np.clip(np.nan_to_num(Jh, nan=1e12, posinf=1e12), 1, 1e12), c=ROJO if diverge else MORADO, lw=2)
    ax1.axhline(J2_OPT, ls="--", c=VERDE, label=f"óptimo J = {J2_OPT:.1f}")
    ax1.set_yscale("log"); ax1.set_xlabel("iteración"); ax1.legend()
    ax1.set_title(("¡DIVERGE!" if diverge else f"J final = {Jh[-1]:,.2f}") + f"  ·  {escala}, α = {alpha:.1e}")
    nombres = FEATURES + ["b"]
    real = np.r_[W2_OPT, B2_OPT]; est = np.r_[w, b] if not diverge else np.full(4, np.nan)
    pct = np.clip(np.nan_to_num(100 * est / real), -50, 200)
    colores = [VERDE if abs(p - 100) < 2 else (NARANJA if abs(p - 100) < 20 else ROJO) for p in pct]
    ax2.bar(nombres, pct, color=colores)
    ax2.axhline(100, ls="--", c="black", lw=1)
    if diverge:
        ax2.text(1.5, 80, "Sin parámetros válidos:\nel algoritmo divergió", ha="center", fontsize=14, color=ROJO, fontweight="bold")
    else:
        for i, p in enumerate(pct):
            ax2.text(i, max(p, 0) + 5, f"{p:.0f}%", ha="center", fontweight="bold")
    ax2.set_ylim(-50, 215); ax2.set_ylabel("% del valor exacto")
    ax2.set_title("Parámetros de GD como % del exacto (línea punteada = 100%)")
    plt.tight_layout(); plt.show()
    if not diverge:
        display(pd.DataFrame({"exacto": real, "GD": est}, index=nombres).round(3).T)

out7 = widgets.interactive_output(_plot_escalado, {"escala": esc_tb, "alpha": a7_sl, "iters": it7_sl})
display(VBox([esc_tb, HBox([a7_sl, it7_sl]), out7]))

## 8. Batch, Mini-batch y Stochastic Gradient Descent

Hasta ahora, en cada paso usamos **todos** los ejemplos para calcular el gradiente (**Batch GD**). Con millones de filas eso es caro. Las alternativas:

| Variante | Ejemplos por paso | Pasos por época | Camino |
|---|---|---|---|
| **Batch GD** | todos ($m$) | 1 | suave y preciso |
| **Mini-batch GD** | un lote (ej. 32, 64, 256) | $m$ / lote | algo ruidoso |
| **Stochastic GD (SGD)** | 1 | $m$ | muy ruidoso, pero avanza rápido |

Una **época** = una pasada completa por los datos. En la práctica (deep learning, XGBoost por dentro, etc.) se usan casi siempre **mini-batches**.

### 🎬 Animación · Las tres variantes con el mismo presupuesto de épocas

In [ ]:
def gd_por_lotes(x, y, alpha, epocas, batch_size, seed=0):
    rng_b = np.random.default_rng(seed); w, b = 0., 0.
    camino = [[(w, b)]]   # lista por época
    for _ in range(epocas):
        idx = rng_b.permutation(len(x)); puntos = []
        for s in range(0, len(x), batch_size):
            lote = idx[s:s + batch_size]
            dw, db = _gradiente(x[lote], y[lote], w, b)
            w, b = w - alpha * dw, b - alpha * db
            puntos.append((w, b))
        camino.append(puntos)
    return camino

EPOCAS, ALPHA_B = 40, 0.3
variantes = {"Batch (24)": (24, AZUL), "Mini-batch (6)": (6, NARANJA), "SGD (1)": (1, ROJO)}
caminos = {k: gd_por_lotes(x_train, y_train, ALPHA_B, EPOCAS, bs) for k, (bs, _) in variantes.items()}

def _acum(camino, e):
    pts = [p for ep in camino[: e + 1] for p in ep]; return np.array(pts)

figb = go.Figure()
figb.add_trace(go.Contour(x=_wg, y=_bg, z=np.log10(JJ), colorscale="Viridis", showscale=False, contours=dict(coloring="lines"), hoverinfo="skip"))
figb.add_trace(go.Scatter(x=[W_OPT], y=[B_OPT], mode="markers", marker=dict(symbol="star", size=16, color=VERDE), name="óptimo"))
for k, (bs, col) in variantes.items():
    P = _acum(caminos[k], 0); figb.add_trace(go.Scatter(x=P[:, 0], y=P[:, 1], mode="lines", line=dict(color=col, width=2), name=k))
figb.frames = [go.Frame(name=str(e), traces=[2, 3, 4], data=[go.Scatter(x=_acum(caminos[k], e)[:, 0], y=_acum(caminos[k], e)[:, 1]) for k in variantes],
                        layout=go.Layout(title_text=f"Época {e} · actualizaciones: Batch={e}, Mini-batch={4*e}, SGD={24*e}")) for e in range(EPOCAS + 1)]
figb.update_layout(title="Época 0", height=560, xaxis=dict(range=W_RANGO, title="w"), yaxis=dict(range=B_RANGO, title="b"),
    updatemenus=[dict(type="buttons", direction="left", x=0, y=-0.1, xanchor="left", buttons=[
        dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=250, redraw=True), fromcurrent=True)]),
        dict(label="⏸ Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.15, y=-0.05, len=0.85, currentvalue=dict(prefix="época: "),
        steps=[dict(label=str(e), method="animate", args=[[str(e)], dict(frame=dict(duration=0, redraw=True), mode="immediate")]) for e in range(EPOCAS + 1)])])
figb.show()

**¿Qué observar?** Con las mismas 40 épocas, Batch GD apenas hizo 40 actualizaciones y sigue lejos; SGD hizo 960 y llegó cerca del óptimo, pero **temblando** alrededor de él. Por eso en SGD se suele **reducir α con el tiempo** (*learning rate schedule*).

## 9. 🤖 Laboratorio con IA: Copilot, Claude, Codex y OpenCode

Ahora te toca dirigir a la IA. Tenemos **funciones de verificación** que comparan cualquier implementación contra la referencia. Úsalas siempre.

In [ ]:
def _check(nombre, ok, detalle=""):
    print(f"{'✅' if ok else '❌'} {nombre} {detalle}")

def verificar_costo(f):
    '''f(x, y, w, b) -> costo'''
    casos = [(0., 0.), (W_OPT, B_OPT), (150., -30.), (310., 75.)]
    ok = all(np.isclose(f(x_train, y_train, w, b), _costo(x_train, y_train, w, b)) for w, b in casos)
    _check("Función de costo", ok)

def verificar_gradiente(f):
    '''f(x, y, w, b) -> (dj_dw, dj_db)'''
    casos = [(0., 0.), (W_OPT, B_OPT), (150., -30.), (310., 75.)]
    ok = all(np.allclose(f(x_train, y_train, w, b), _gradiente(x_train, y_train, w, b), atol=1e-6) for w, b in casos)
    _check("Gradiente", ok)

def verificar_gd(f):
    '''f(x, y, w_in, b_in, alpha, num_iters) -> (w, b, ...)'''
    res = f(x_train, y_train, 0., 0., 0.5, 1000)
    w, b = res[0], res[1]
    ok = np.isclose(w, W_OPT, atol=1e-2) and np.isclose(b, B_OPT, atol=1e-2)
    _check("Gradient descent", ok, f"→ obtuvo w={w:.3f}, b={b:.3f} (esperado w={W_OPT:.3f}, b={B_OPT:.3f})")

# Probamos las verificaciones con nuestras propias funciones
verificar_costo(compute_cost)
verificar_gradiente(compute_gradient)
verificar_gd(lambda x, y, w, b, a, n: gradient_descent(x, y, w, b, a, n, compute_cost, compute_gradient, verbose=False))

### Reto 1 · 🕵️ Encuentra el bug (código "generado por IA")

Una IA nos entregó esta función. Se ve razonable, corre sin errores… **pero está mal**. Ejecútala, mira la verificación y encuentra los problemas. Luego **pídele a Claude** que la revise y compara su diagnóstico con el tuyo.

In [ ]:
def gradient_descent_ia(x, y, w_in, b_in, alpha, num_iters):
    w, b = w_in, b_in
    for i in range(num_iters):
        err = w * x + b - y
        dj_dw = np.mean(err) * np.mean(x)
        w = w - alpha * dj_dw
        err = w * x + b - y
        dj_db = np.mean(err)
        b = b - alpha * dj_db
    return w, b

verificar_gd(gradient_descent_ia)

<details>
<summary>👀 Ver solución (después de intentarlo)</summary>

1. `dj_dw = np.mean(err) * np.mean(x)` está mal: la derivada es `np.mean(err * x)`. **La media de un producto no es el producto de las medias.** Con este error el algoritmo se detiene apenas el error *promedio* es cero, aunque la pendiente sea incorrecta.
2. La actualización **no es simultánea**: recalcula `err` con el `w` nuevo antes de calcular `dj_db`.

Moraleja: el código "corre" ≠ el código "es correcto". Por eso verificamos.
</details>

### Reto 2 · ⌨️ Copilot: de comentario a código

En la celda de abajo, escribe solo el comentario y deja que **Copilot** complete la función (en VS Code o Colab, presiona `Tab` para aceptar la sugerencia). Después ejecuta la verificación.

```python
# Implementa compute_gradient_ia(x, y, w, b) para regresión lineal univariada:
# devuelve (dj_dw, dj_db) promediando sobre los m ejemplos, vectorizado con numpy, sin bucles for.
```

In [ ]:
# ✍️ Escribe aquí el comentario y deja que Copilot complete la función



# verificar_gradiente(compute_gradient_ia)

### Reto 3 · 💬 Claude: explicar y razonar

Copia este prompt en Claude (puedes adaptarlo):

> *Estoy aprendiendo gradient descent para regresión lineal con una variable. Mis datos tienen mean(x) ≈ 1.05 y mean(x²) ≈ 1.22. Con learning rate 0.9 converge, pero con 0.95 diverge. Explícame por qué usando la matriz Hessiana y sus autovalores, y dime cuál es el learning rate máximo. Luego dame una analogía para explicárselo a alguien sin formación matemática.*

Después **verifica su respuesta** con la celda de autovalores de la sección 6. ¿Coincide el α máximo que calculó?

### Reto 4 · 🛠️ Codex u OpenCode: un módulo con tests

Abre una terminal en una carpeta nueva y dale esta tarea a **Codex** o a **OpenCode**:

> *Crea un paquete Python `gd_lab` con un archivo `gd.py` que contenga `compute_cost`, `compute_gradient` y `gradient_descent` para regresión lineal múltiple (vectorizado con NumPy). `gradient_descent` debe aceptar `tol` para detenerse cuando la mejora del costo sea menor a `tol`, y devolver el historial de costos. Crea `tests/test_gd.py` con pytest que compare el resultado con `np.linalg.lstsq` en datos sintéticos, pruebe el gradiente con diferencias finitas y verifique que con un learning rate muy grande el costo aumenta. Ejecuta los tests y corrige hasta que pasen.*

**Duelo de IAs 🥊:** la mitad del salón usa Codex y la otra mitad OpenCode (o el modelo que prefieran). Comparamos:
- ¿Pasaron todos los tests a la primera?
- ¿Cuántas iteraciones de corrección necesitó el agente?
- ¿El código es legible? ¿Respetó la actualización simultánea?

Luego importa tu función aquí y pásala por `verificar_gd`.

In [ ]:
# from gd_lab.gd import gradient_descent as gd_agente
# verificar_gd(lambda x, y, w, b, a, n: gd_agente(x.reshape(-1, 1), y, np.array([w]), b, a, n)[:2])

### ✅ Checklist para auditar código de ML generado por IA

- [ ] ¿La fórmula coincide con la matemática? (signo, $\frac{1}{m}$ vs $\frac{1}{2m}$, producto con $x$)
- [ ] ¿La actualización de parámetros es **simultánea**?
- [ ] ¿Pasa el **gradient checking** numérico?
- [ ] ¿Coincide con una referencia confiable? (`sklearn`, `np.linalg.lstsq`)
- [ ] ¿La curva de aprendizaje **baja**? ¿Qué pasa con un α grande?
- [ ] ¿El escalamiento usa estadísticas **solo de entrenamiento**?
- [ ] ¿Está vectorizado o tiene bucles innecesarios?

## 10. Resumen

| Concepto | Lo esencial |
|---|---|
| Modelo | $f_{w,b}(x) = wx + b$; entrenar = encontrar $w$ y $b$. |
| Costo | $J(w,b)=\frac{1}{2m}\sum(f-y)^2$; en regresión lineal es un tazón convexo con un solo mínimo. |
| Gradiente | Pendiente de $J$; el paso va en dirección contraria. Se audita con diferencias finitas. |
| Gradient Descent | $w \leftarrow w - \alpha\,\partial J/\partial w$, $b \leftarrow b - \alpha\,\partial J/\partial b$, actualización simultánea. |
| Learning rate | Pequeño = lento; grande = zig-zag; demasiado grande = diverge ($\alpha < 2/\lambda_{\max}$). |
| Escalamiento | Las variables en escalas distintas alargan el tazón; z-score lo redondea y acelera GD. |
| Variantes | Batch (estable), Mini-batch (el estándar), SGD (rápido y ruidoso). |
| IA | La IA escribe, **tú verificas**. |

### 📝 Tarea

1. Agrega a `gradient_descent_multi` un criterio de **parada temprana** (`tol`) y un **learning rate schedule** ($\alpha_t = \alpha_0 / (1 + k\,t)$). Puedes usar la IA que prefieras.
2. Entrena el modelo de 3 variables con y sin escalamiento y reporta cuántas iteraciones necesitó cada uno para llegar a 1% del costo óptimo.
3. Escribe en una celda Markdown **qué IA usaste, qué prompt le diste y qué error tuvo que corregir** (si no tuvo ninguno, explica cómo lo verificaste).

**Próxima clase (lunes 30/11):** Regresión Logística avanzada y Regularización (Lasso, Ridge, ElasticNet): veremos que el mismo gradient descent sirve para clasificar y cómo evitar el sobreajuste.